In [1]:
!ls /kaggle/input/datasets/kamikazeeeeeee88/
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1 
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/images_train| head -n 5
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/labels_train| head -n 5
!cat /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/labels_train/Clip_001_00056.txt




nps-drone-test	nps-drone-train-part1  nps-drone-train-part2  nps-drone-val
images_train  labels_train
Clip_001_00056.png
Clip_001_00057.png
Clip_001_00058.png
Clip_001_00059.png
Clip_001_00060.png
ls: write error: Broken pipe
Clip_001_00056.txt
Clip_001_00057.txt
Clip_001_00058.txt
Clip_001_00059.txt
Clip_001_00060.txt
ls: write error: Broken pipe
0 0.560937 0.184259 0.005208 0.009259


In [2]:
# =============================================================================
# CELL 1 — Install dependencies
# =============================================================================
import subprocess, sys

pkgs = [
    "wandb",
    "timm",
    "albumentations>=1.4.0",
    "torchmetrics",
    "pycocotools",
    "einops",
    "tqdm",
    "matplotlib",
    "seaborn",
    "pandas",
    "numpy",
    "opencv-python-headless",
    "scipy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.4 MB/s eta 0:00:00
All packages installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [3]:
# =============================================================================
# CELL 2 — Imports
# =============================================================================
import os, json, math, time, random, glob, copy, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import maximum_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.ops as tv_ops

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import wandb
from kaggle_secrets import UserSecretsClient

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
print("Imports done. Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Imports done. Torch: 2.10.0+cu128 | CUDA: True


In [6]:
# =============================================================================
# CELL 3 — Global configuration (single source of truth)
# =============================================================================
CFG = {
    # ── project ──────────────────────────────────────────────────────────────
    "project"        : "drone-detection-nps",
    "run_name"       : "heatmap-guided-single-anchor-motion-v3",
    "seed"           : 42,

    # ── data ─────────────────────────────────────────────────────────────────
    "data_root"      : "/kaggle/input/datasets/kamikazeeeeeee88",
    "train_parts"    : ["nps-drone-train-part1", "nps-drone-train-part2"],
    "val_dir"        : "nps-drone-val",
    "test_dir"       : "nps-drone-test",
    "img_size"       : 640,
    "temporal_len"   : 3,

    # ── heatmap ──────────────────────────────────────────────────────────────
    "heatmap_stride" : 4,
    "heatmap_sigma"  : 2.0,
    "topk"           : 50,

    # ── backbone ─────────────────────────────────────────────────────────────
    "backbone"       : "efficientnet_b0",
    "pretrained"     : True,
    "backbone_out_channels": [40, 112, 320],

    # ── BiFPN ────────────────────────────────────────────────────────────────
    "bifpn_channels" : 64,
    "bifpn_layers"   : 3,

    # ── motion channels (kept — this is the component you want retained) ────
    "use_motion"        : True,
    "motion_channels"   : 2,

    # ── ROIAlign ─────────────────────────────────────────────────────────────
    "roi_output_size": 11,
    "roi_levels"     : [0, 1, 2],

    # ── single DATA-DRIVEN anchor (replaces both the old fixed-16px single
    #    anchor AND the multi-anchor head). The actual value is computed
    #    from your training labels in Cell 5 and written into this dict
    #    before the model is built — "anchor_size" stays None here as a
    #    placeholder so it's obvious if something forgets to set it.
    "anchor_size"    : None,
    "refine_hidden"  : 256,
    "num_classes"    : 1,

    # ── training ─────────────────────────────────────────────────────────────
    "epochs"         : 60,
    "batch_size"     : 4,
    "accum_steps"    : 4,
    "num_workers"    : 2,
    "lr"             : 2e-4,
    "weight_decay"   : 1e-4,
    "warmup_epochs"  : 5,
    "cosine_min_lr"  : 1e-6,
    "clip_grad"      : 2.0,

    # ── loss weights ─────────────────────────────────────────────────────────
    "heatmap_loss_w" : 1.0,
    "obj_loss_w"     : 1.0,
    "box_loss_w"     : 2.0,

    # ── focal loss params ────────────────────────────────────────────────────
    "focal_alpha"    : 2.0,
    "focal_beta"     : 4.0,

    # ── IoU assignment (tuned for tiny objects) ───────────────────────────────
    "iou_pos_thr"    : 0.25,
    "iou_neg_thr"    : 0.05,

    # ── NMS ──────────────────────────────────────────────────────────────────
    "nms_iou_thr"    : 0.3,
    "conf_thr"       : 0.3,

    # ── epoch subsampling (NEW) ─────────────────────────────────────────────
    # Each epoch trains on a freshly-drawn random subset of the full
    # training set rather than the whole thing. Over many epochs this
    # approaches full coverage while keeping any single epoch cheap —
    # this is the lever for fitting more epochs into limited days.
    "epoch_subsample_fraction": 1.0,   # 35% of train_triplets per epoch

    # ── misc ─────────────────────────────────────────────────────────────────
    "save_dir"       : "/kaggle/working/checkpoints",
    "amp"            : True,
}

os.makedirs(CFG["save_dir"], exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Config set. (anchor_size will be computed from data in Cell 5)")

Device: cuda
Config set. (anchor_size will be computed from data in Cell 5)


In [25]:
# =============================================================================
# CELL 4 — W&B initialisation
# =============================================================================
secrets   = UserSecretsClient()
WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_KEY)

run = wandb.init(
    project   = CFG["project"],
    name      = CFG["run_name"],
    config    = CFG,
    save_code = True,
)
print(f"W&B run: {run.url}")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B run: https://wandb.ai/bscs22030-information-technology-university/drone-detection-nps/runs/0kvv98cx


In [10]:
# =============================================================================
# CELL 5 — Dataset preparation: build file-lists, COCO JSON, and compute
# the data-driven anchor size from actual training-set box statistics
# =============================================================================

def yolo_to_abs(cx, cy, w, h, W, H):
    cx, cy, w, h = cx * W, cy * H, w * W, h * H
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2


def load_clip_index(image_dir, label_dir):
    img_paths = sorted(glob.glob(str(image_dir / "*.png")) +
                       glob.glob(str(image_dir / "*.jpg")))
    pairs = []
    for ip in img_paths:
        stem = Path(ip).stem
        lp   = label_dir / f"{stem}.txt"
        pairs.append((ip, str(lp) if lp.exists() else None))
    return pairs


def build_temporal_triplets(pairs, T=3):
    clips = defaultdict(list)
    for ip, lp in pairs:
        clip_id = "_".join(Path(ip).stem.split("_")[:2])
        clips[clip_id].append((ip, lp))

    triplets = []
    for clip_id in sorted(clips.keys()):
        frames = clips[clip_id]
        for i in range(len(frames) - T + 1):
            triplets.append(frames[i : i + T])
    return triplets


def build_dataset_splits():
    root = Path(CFG["data_root"])
    train_triplets = []
    for part in CFG["train_parts"]:
        img_dir = root / part / "images_train"
        lbl_dir = root / part / "labels_train"
        pairs   = load_clip_index(img_dir, lbl_dir)
        train_triplets.extend(build_temporal_triplets(pairs, CFG["temporal_len"]))

    val_img = root / CFG["val_dir"] / "images_val"
    val_lbl = root / CFG["val_dir"] / "labels_val"
    val_triplets = build_temporal_triplets(
        load_clip_index(val_img, val_lbl), CFG["temporal_len"])

    test_img = root / CFG["test_dir"] / "images_test"
    test_lbl = root / CFG["test_dir"] / "labels_test"
    test_triplets = build_temporal_triplets(
        load_clip_index(test_img, test_lbl), CFG["temporal_len"])

    print(f"Train triplets : {len(train_triplets)}")
    print(f"Val   triplets : {len(val_triplets)}")
    print(f"Test  triplets : {len(test_triplets)}")
    return train_triplets, val_triplets, test_triplets


def read_labels(lbl_path):
    if lbl_path is None or not os.path.exists(lbl_path):
        return []
    boxes = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                boxes.append([float(x) for x in parts[1:5]])
    return boxes


def build_coco_json(triplets, out_path, img_size):
    images, annotations = [], []
    ann_id = 1
    for img_id, triplet in enumerate(triplets):
        img_path, lbl_path = triplet[-1]
        img = cv2.imread(img_path)
        H, W = img.shape[:2]

        scale    = img_size / max(H, W)
        new_H    = int(round(H * scale))
        new_W    = int(round(W * scale))
        pad_top  = (img_size - new_H) // 2
        pad_left = (img_size - new_W) // 2

        images.append({"id": img_id, "file_name": img_path,
                        "width": img_size, "height": img_size})

        for cx, cy, bw, bh in read_labels(lbl_path):
            x1, y1, x2, y2 = yolo_to_abs(cx, cy, bw, bh, W, H)
            x1 = x1 * scale + pad_left
            y1 = y1 * scale + pad_top
            x2 = x2 * scale + pad_left
            y2 = y2 * scale + pad_top
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "area": (x2 - x1) * (y2 - y1), "iscrowd": 0,
            })
            ann_id += 1

    coco_dict = {"images": images, "annotations": annotations,
                "categories": [{"id": 1, "name": "drone"}]}
    with open(out_path, "w") as f:
        json.dump(coco_dict, f)
    print(f"Saved COCO JSON → {out_path}  ({len(images)} imgs, {len(annotations)} anns)")
    return out_path


def compute_data_driven_anchor_size(triplets, img_size, sample_limit=2000):
    """
    Computes the median GT box dimension (in 640-space pixels) across the
    training set, to use as a single data-driven anchor prior — this
    replaces the old hand-picked fixed value (16px, then 8px) with a
    value derived directly from the actual target size distribution,
    addressing the 'fixed anchor is a limitation' concern without the
    complexity (and instability) of a multi-anchor head.
    """
    sizes = []
    for triplet in triplets[:sample_limit]:
        _, lp = triplet[-1]
        for cx, cy, bw, bh in read_labels(lp):
            sizes.append(bw * img_size)
            sizes.append(bh * img_size)
    if len(sizes) == 0:
        print("WARNING: no GT boxes found for anchor computation, defaulting to 8.0")
        return 8.0
    median_size = float(np.median(sizes))
    p25 = float(np.percentile(sizes, 25))
    p75 = float(np.percentile(sizes, 75))
    print(f"GT box size distribution @ {img_size}px (n={len(sizes)} measurements): "
          f"p25={p25:.2f}, median={median_size:.2f}, p75={p75:.2f}")
    return median_size


train_triplets, val_triplets, test_triplets = build_dataset_splits()

# ── compute and lock in the data-driven anchor size ──────────────────────
CFG["anchor_size"] = compute_data_driven_anchor_size(
    train_triplets, CFG["img_size"])
print(f"Data-driven anchor_size set to: {CFG['anchor_size']:.2f}px")

VAL_COCO_JSON  = "/kaggle/working/val_coco.json"
TEST_COCO_JSON = "/kaggle/working/test_coco.json"
build_coco_json(val_triplets,  VAL_COCO_JSON,  CFG["img_size"])
build_coco_json(test_triplets, TEST_COCO_JSON, CFG["img_size"])

Train triplets : 32148
Val   triplets : 3745
Test  triplets : 12335
GT box size distribution @ 640px (n=4000 measurements): p25=3.33, median=5.93, p75=5.93
Data-driven anchor_size set to: 5.93px
Saved COCO JSON → /kaggle/working/val_coco.json  (3745 imgs, 4648 anns)
Saved COCO JSON → /kaggle/working/test_coco.json  (12335 imgs, 16360 anns)


'/kaggle/working/test_coco.json'

In [11]:
# =============================================================================
# CELL 5.5 — Anchor-size consistency check + auto-recovery
# Run AFTER Cell 5 (data prep) and AFTER Cell 17.8 (checkpoint pull),
# BEFORE Cell 11 (model build).
# =============================================================================

ckpt_path_to_check = os.path.join(CFG["save_dir"], "checkpoint_last.pt")

assert os.path.exists(ckpt_path_to_check), (
    f"Checkpoint not found at {ckpt_path_to_check} — pull it from W&B "
    f"(Cell 17.8) before running this cell.")

ckpt_inspect = torch.load(ckpt_path_to_check, map_location="cpu")
saved_cfg    = ckpt_inspect.get("cfg", {})
old_anchor   = saved_cfg.get("anchor_size")
new_anchor   = CFG["anchor_size"]

print(f"Checkpoint epoch          : {ckpt_inspect.get('epoch')}")
print(f"Anchor size used in training (from checkpoint's saved cfg): {old_anchor}")
print(f"Anchor size this session's Cell 5 just computed            : {new_anchor}")

if old_anchor is None:
    print("⚠ WARNING: checkpoint has no 'anchor_size' in its saved cfg. "
          "Proceeding with this session's freshly computed value.")
elif abs(old_anchor - new_anchor) < 1e-4:
    print(f"✅ MATCH — anchor_size is consistent ({new_anchor:.6f}).")
else:
    CFG["anchor_size"] = old_anchor
    print(f"❌ MISMATCH — overriding CFG['anchor_size'] to {old_anchor:.6f} "
          f"(was about to use {new_anchor:.6f}).")

del ckpt_inspect

Checkpoint epoch          : 30
Anchor size used in training (from checkpoint's saved cfg): 5.92576
Anchor size this session's Cell 5 just computed            : 5.92576
✅ MATCH — anchor_size is consistent (5.925760).


In [12]:
# =============================================================================
# CELL 6 — Augmentation pipelines (temporally consistent, motion-aware)
# =============================================================================

PIXEL_TRAIN = A.Compose([
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50)),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1)),
    ], p=0.6),
    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=(3, 5)),
    ], p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                         val_shift_limit=20, p=0.4),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.2),
    A.ToFloat(max_value=255.0),
])

PIXEL_VAL = A.Compose([A.ToFloat(max_value=255.0)])

SPATIAL_TRAIN = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2,
                       rotate_limit=15, border_mode=cv2.BORDER_CONSTANT,
                       value=0, p=0.6),
    A.RandomCrop(height=CFG["img_size"], width=CFG["img_size"], p=0.3),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
],
    bbox_params=A.BboxParams(
        format="yolo", label_fields=["class_labels"],
        min_visibility=0.3, 
    )
)

SPATIAL_VAL = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
],
    bbox_params=A.BboxParams(
        format="yolo", label_fields=["class_labels"],
        min_visibility=0.3, 
    )
)


def sanitise_yolo_boxes(boxes):
    clean = []
    for box in boxes:
        cx, cy, bw, bh = box
        eps = 1e-6
        bw  = float(np.clip(bw, eps, 1.0))
        bh  = float(np.clip(bh, eps, 1.0))
        cx  = float(np.clip(cx, bw / 2, 1.0 - bw / 2))
        cy  = float(np.clip(cy, bh / 2, 1.0 - bh / 2))
        clean.append((cx, cy, bw, bh))
    return clean


def apply_temporal_augmentation(frames_bgr, boxes_list, is_train):
    spatial_tf = SPATIAL_TRAIN if is_train else SPATIAL_VAL
    pixel_tf   = PIXEL_TRAIN   if is_train else PIXEL_VAL

    anchor_boxes_raw = sanitise_yolo_boxes(boxes_list[-1])
    triplet_seed = random.randint(0, 2**31 - 1)

    processed_frames = []
    aligned_gray_frames = []
    final_boxes = anchor_boxes_raw

    for i, frame in enumerate(frames_bgr):
        is_anchor = (i == len(frames_bgr) - 1)
        boxes     = anchor_boxes_raw if is_anchor else []
        labels    = [0] * len(boxes)

        random.seed(triplet_seed)
        np.random.seed(triplet_seed)
        out = spatial_tf(image=frame, bboxes=boxes, class_labels=labels)

        if is_anchor:
            final_boxes = sanitise_yolo_boxes(list(out["bboxes"]))

        aligned_rgb = out["image"]
        aligned_gray = cv2.cvtColor(aligned_rgb, cv2.COLOR_RGB2GRAY)
        aligned_gray = aligned_gray.astype(np.float32) / 255.0
        aligned_gray_frames.append(aligned_gray)

        out2 = pixel_tf(image=out["image"])
        processed_frames.append(out2["image"])

    frames_np = np.stack(processed_frames, axis=0)
    frames_t  = torch.from_numpy(frames_np.transpose(0, 3, 1, 2))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    frames_t = (frames_t - mean) / std

    gray_np = np.stack(aligned_gray_frames, axis=0)
    gray_t  = torch.from_numpy(gray_np)

    return frames_t, final_boxes, gray_t


print("Augmentation pipelines ready.")

Augmentation pipelines ready.


In [13]:
# =============================================================================
# CELL 7 — Dataset, DataLoader, and epoch-subsampling builder
# =============================================================================

def gaussian2D(shape, sigma=1.0):
    m, n   = [(s - 1.) / 2. for s in shape]
    y, x   = np.ogrid[-m:m+1, -n:n+1]
    kernel = np.exp(-(x * x + y * y) / (2 * sigma * sigma))
    kernel[kernel < np.finfo(kernel.dtype).eps * kernel.max()] = 0
    return kernel


def build_gaussian_heatmap(centers_xy, H, W, sigma=2.0):
    hm = np.zeros((H, W), dtype=np.float32)
    if not centers_xy:
        return hm
    radius   = max(int(math.ceil(3 * sigma)), 1)
    diameter = 2 * radius + 1
    kernel   = gaussian2D((diameter, diameter), sigma=sigma)
    kernel[radius, radius] = 1.0
    for (cx, cy) in centers_xy:
        ix, iy = int(round(cx)), int(round(cy))
        x0h = max(0,  ix - radius);   x1h = min(W, ix + radius + 1)
        y0h = max(0,  iy - radius);   y1h = min(H, iy + radius + 1)
        x0k = x0h - (ix - radius);    x1k = x0k + (x1h - x0h)
        y0k = y0h - (iy - radius);    y1k = y0k + (y1h - y0h)
        if x1h <= x0h or y1h <= y0h:
            continue
        hm[y0h:y1h, x0h:x1h] = np.maximum(
            hm[y0h:y1h, x0h:x1h], kernel[y0k:y1k, x0k:x1k])
    return hm


def compute_motion_diffs(gray_t):
    T = gray_t.shape[0]
    anchor = gray_t[-1]
    diffs = []
    for i in range(T - 1):
        diffs.append(torch.abs(anchor - gray_t[i]))
    return torch.stack(diffs, dim=0)


class DroneDataset(Dataset):
    def __init__(self, triplets, is_train=True):
        self.triplets = triplets
        self.is_train = is_train
        self.HM_H = CFG["img_size"] // CFG["heatmap_stride"]
        self.HM_W = CFG["img_size"] // CFG["heatmap_stride"]

    def __len__(self):
        return len(self.triplets)

    def _load_frame(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        triplet    = self.triplets[idx]
        frames_bgr = [self._load_frame(ip) for ip, _ in triplet]
        boxes_list = [read_labels(lp) for _, lp in triplet]

        frames_t, anchor_boxes, gray_t = apply_temporal_augmentation(
            frames_bgr, boxes_list, self.is_train)

        motion_t = compute_motion_diffs(gray_t) if CFG["use_motion"] else None

        centers      = []
        gt_boxes_abs = []
        for (cx, cy, bw, bh) in anchor_boxes:
            hm_x = cx * self.HM_W
            hm_y = cy * self.HM_H
            centers.append((hm_x, hm_y))

            px = cx * CFG["img_size"]
            py = cy * CFG["img_size"]
            pw = bw * CFG["img_size"]
            ph = bh * CFG["img_size"]
            gt_boxes_abs.append([px - pw/2, py - ph/2, px + pw/2, py + ph/2])

        heatmap   = build_gaussian_heatmap(
            centers, self.HM_H, self.HM_W, CFG["heatmap_sigma"])
        heatmap_t = torch.from_numpy(heatmap).unsqueeze(0)

        max_objs = 64
        gt_arr   = np.zeros((max_objs, 4), dtype=np.float32)
        gt_mask  = np.zeros(max_objs,      dtype=np.float32)
        for i, box in enumerate(gt_boxes_abs[:max_objs]):
            gt_arr[i]  = box
            gt_mask[i] = 1.0

        item = {
            "frames"  : frames_t,
            "heatmap" : heatmap_t,
            "gt_boxes": torch.from_numpy(gt_arr),
            "gt_mask" : torch.from_numpy(gt_mask),
            "img_id"  : idx,
        }
        if CFG["use_motion"]:
            item["motion"] = motion_t
        return item


def get_epoch_subset_dataloader(full_triplets, fraction, batch_size, num_workers):
    """
    Draws a NEW random subset of the full training set each call. Intended
    to be called once per epoch with a fresh draw, so that over many
    epochs the model approaches full-dataset coverage while any single
    epoch only costs `fraction` of full-dataset wall-clock time.
    """
    n_subset = max(int(len(full_triplets) * fraction), batch_size)
    subset_triplets = random.sample(full_triplets, n_subset)
    subset_ds = DroneDataset(subset_triplets, is_train=True)
    subset_dl = DataLoader(subset_ds, batch_size=batch_size,
                           shuffle=True, num_workers=num_workers,
                           pin_memory=True, drop_last=True)
    return subset_dl, n_subset


def build_static_dataloaders(val_triplets, test_triplets):
    """Val/test stay FULL and FIXED every epoch — only train subsamples."""
    val_ds   = DroneDataset(val_triplets,  is_train=False)
    test_ds  = DroneDataset(test_triplets, is_train=False)

    val_dl   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
    return val_dl, test_dl


val_dl, test_dl = build_static_dataloaders(val_triplets, test_triplets)

# initial train_dl draw, so cells that need len(train_dl) (e.g. optimizer
# step-count calc) have something to read before the first real epoch
train_dl, _n0 = get_epoch_subset_dataloader(
    train_triplets, CFG["epoch_subsample_fraction"],
    CFG["batch_size"], CFG["num_workers"])

print(f"Val   batches : {len(val_dl)} (full, fixed)")
print(f"Test  batches : {len(test_dl)} (full, fixed)")
print(f"Train batches : {len(train_dl)} "
      f"({CFG['epoch_subsample_fraction']*100:.0f}% subsample, "
      f"redrawn fresh each epoch)")

Val   batches : 937 (full, fixed)
Test  batches : 3084 (full, fixed)
Train batches : 8037 (100% subsample, redrawn fresh each epoch)


In [28]:
# =============================================================================
# CELL 9-11 — Full HeatmapGuidedDetector model
# (single data-driven anchor + motion fusion, multi-anchor head removed)
# =============================================================================

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, kernel, padding=padding,
                             groups=in_ch, bias=False)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class TemporalFusion(nn.Module):
    def __init__(self, ch, T):
        super().__init__()
        self.T    = T
        self.fuse = nn.Sequential(
            nn.Conv2d(ch * T, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        BT, C, H, W = x.shape
        B = BT // self.T
        return self.fuse(x.view(B, self.T * C, H, W))


class BiFPNNode(nn.Module):
    def __init__(self, ch, n_inputs=2):
        super().__init__()
        self.w   = nn.Parameter(torch.ones(n_inputs, dtype=torch.float32))
        self.eps = 1e-4
        self.conv= DepthwiseSeparableConv(ch, ch)

    def forward(self, feats):
        w = F.relu(self.w)
        w = w / (w.sum() + self.eps)
        fused = sum(w[i] * feats[i] for i in range(len(feats)))
        return self.conv(fused)


class BiFPNLayer(nn.Module):
    def __init__(self, ch, num_levels=3):
        super().__init__()
        self.td_nodes = nn.ModuleList(
            [BiFPNNode(ch, n_inputs=2) for _ in range(num_levels - 1)])
        self.bu_nodes = nn.ModuleList(
            [BiFPNNode(ch, n_inputs=3 if i < num_levels - 2 else 2)
             for i in range(num_levels - 1)])

    def forward(self, features):
        n = len(features)
        td = [None] * n
        td[-1] = features[-1]
        for i in range(n - 2, -1, -1):
            up    = F.interpolate(td[i + 1], size=features[i].shape[-2:],
                                  mode="nearest")
            td[i] = self.td_nodes[i]([features[i], up])
        out = [None] * n
        out[0] = td[0]
        for i in range(1, n):
            dn = F.adaptive_avg_pool2d(out[i - 1], features[i].shape[-2:])
            if i < n - 1:
                out[i] = self.bu_nodes[i - 1]([features[i], td[i], dn])
            else:
                out[i] = self.bu_nodes[i - 1]([features[i], dn])
        return out


class BiFPN(nn.Module):
    def __init__(self, in_channels_list, out_ch, num_layers=3):
        super().__init__()
        self.lateral = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.SiLU(inplace=True))
            for c in in_channels_list
        ])
        self.layers = nn.ModuleList([
            BiFPNLayer(out_ch, num_levels=3)
            for _ in range(num_layers)
        ])
        self.p1_upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            DepthwiseSeparableConv(out_ch, out_ch),
        )

    def forward(self, c_feats):
        feats = [lat(c) for lat, c in zip(self.lateral, c_feats)]
        for layer in self.layers:
            feats = layer(feats)
        p1 = self.p1_upsample(feats[0])
        return [p1] + feats


class MotionEncoder(nn.Module):
    """
    Downsample-first motion encoder: downsamples raw motion-difference
    maps to the target (P1) resolution BEFORE running convs — far
    cheaper than running convs at full image resolution, with
    negligible quality cost for tiny-target motion coherence.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(in_ch, out_ch // 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch // 2),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch // 2, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, motion, target_size):
        motion_ds = F.adaptive_avg_pool2d(motion, target_size)
        return self.enc(motion_ds)


class HeatmapHead(nn.Module):
    def __init__(self, in_ch, num_classes=1, use_motion=True):
        super().__init__()
        self.use_motion = use_motion
        if use_motion:
            self.fuse_conv = nn.Sequential(
                nn.Conv2d(in_ch * 2, in_ch, 1, bias=False),
                nn.BatchNorm2d(in_ch),
                nn.SiLU(inplace=True),
            )
        self.head = nn.Sequential(
            DepthwiseSeparableConv(in_ch, in_ch),
            DepthwiseSeparableConv(in_ch, in_ch),
            nn.Conv2d(in_ch, num_classes, 1),
        )

    def forward(self, p1, motion_feat=None):
        if self.use_motion and motion_feat is not None:
            x = torch.cat([p1, motion_feat], dim=1)
            x = self.fuse_conv(x)
        else:
            x = p1
        return self.head(x)


def extract_peaks(heatmap_logits, topk=50, kernel=3, conf_thr=0.01):
    B    = heatmap_logits.shape[0]
    heat = torch.sigmoid(heatmap_logits)
    pad  = kernel // 2
    hmax = F.max_pool2d(heat, kernel, stride=1, padding=pad)
    keep = (heat == hmax).float() * heat

    peaks_list = []
    for b in range(B):
        flat = keep[b, 0].reshape(-1)
        k    = min(topk, int((flat > conf_thr).sum().item()))
        k    = max(k, 1)
        vals, inds = flat.topk(k)
        H, W  = keep.shape[-2:]
        ys    = inds // W
        xs    = inds %  W
        coords = torch.stack([ys, xs], dim=1).float()
        peaks_list.append((coords, vals))
    return peaks_list


class RefinementHead(nn.Module):
    """Single-anchor refinement head (no multi-anchor selection logic)."""
    def __init__(self, roi_size, num_fpn_levels, fpn_ch, hidden):
        super().__init__()
        flat_dim = num_fpn_levels * fpn_ch * roi_size * roi_size
        self.mlp = nn.Sequential(
            nn.Linear(flat_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.SiLU(inplace=True),
        )
        self.obj_head = nn.Linear(hidden // 2, 1)
        self.box_head = nn.Linear(hidden // 2, 4)

    def forward(self, roi_feats):
        h   = self.mlp(roi_feats)
        obj = self.obj_head(h)
        box = self.box_head(h)
        return obj, box


class HeatmapGuidedDetector(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        T        = cfg["temporal_len"]
        fpn_ch   = cfg["bifpn_channels"]
        roi_size = cfg["roi_output_size"]

        self.backbone = timm.create_model(
            cfg["backbone"], pretrained=cfg["pretrained"],
            features_only=True, out_indices=(2, 3, 4))
        bb_ch = cfg["backbone_out_channels"]

        self.temporal_fuse = nn.ModuleList(
            [TemporalFusion(c, T) for c in bb_ch])
        self.bifpn = BiFPN(bb_ch, fpn_ch, cfg["bifpn_layers"])

        self.use_motion = cfg["use_motion"]
        if self.use_motion:
            self.motion_encoder = MotionEncoder(cfg["motion_channels"], fpn_ch)

        self.hm_head = HeatmapHead(fpn_ch, cfg["num_classes"], use_motion=self.use_motion)

        roi_levels   = [0, 1, 2]
        self.roi_levels = roi_levels

        self.refine_head = RefinementHead(
            roi_size, len(roi_levels), fpn_ch, cfg["refine_hidden"])

        self.roi_size    = roi_size
        self.fpn_ch      = fpn_ch
        self.hm_stride   = 4
        self.topk        = cfg["topk"]
        self.img_size    = cfg["img_size"]

        # ── single data-driven anchor (computed in Cell 5 from real GT stats) ──
        assert cfg["anchor_size"] is not None, (
            "CFG['anchor_size'] is None — Cell 5 must run and compute it "
            "from training labels before the model is built.")
        self.anchor_size = cfg["anchor_size"]

    def _extract_backbone(self, frames):
        B, T, C, H, W = frames.shape
        x     = frames.view(B * T, C, H, W)
        feats = self.backbone(x)
        fused = [self.temporal_fuse[i](feats[i]) for i in range(3)]
        return fused

    def _roi_align_single(self, fpn_feats, peaks_batch, B):
        all_roi_feats = []
        all_anchors   = []
        batch_splits  = []

        for b in range(B):
            coords, scores = peaks_batch[b]
            N = coords.shape[0]
            batch_splits.append(N)

            cxs = coords[:, 1] * self.hm_stride
            cys = coords[:, 0] * self.hm_stride

            level_roi_list = []
            for lvl_idx in self.roi_levels:
                fmap     = fpn_feats[lvl_idx]
                h_f, w_f = fmap.shape[-2:]
                scale_x  = w_f / self.img_size
                scale_y  = h_f / self.img_size

                fc_x = cxs * scale_x
                fc_y = cys * scale_y
                half = self.roi_size / 2.0

                x1_f = fc_x - half; y1_f = fc_y - half
                x2_f = fc_x + half; y2_f = fc_y + half

                batch_col = torch.full((N, 1), float(b), device=fmap.device)
                rois_in = torch.stack(
                    [batch_col[:, 0], x1_f, y1_f, x2_f, y2_f], dim=1)

                roi_out = tv_ops.roi_align(
                    fmap, rois_in, output_size=self.roi_size,
                    spatial_scale=1.0, sampling_ratio=2, aligned=True)
                level_roi_list.append(roi_out)

            cat_roi = torch.cat(level_roi_list, dim=1)
            all_roi_feats.append(cat_roi.view(N, -1))

            anchors_b = torch.stack([
                cxs, cys,
                torch.full_like(cxs, self.anchor_size),
                torch.full_like(cys, self.anchor_size),
            ], dim=1)
            all_anchors.append(anchors_b)

        all_roi_feats_cat = torch.cat(all_roi_feats, dim=0)
        return all_roi_feats_cat, all_anchors, batch_splits

    def forward(self, frames, motion=None):
        B         = frames.shape[0]
        c_feats   = self._extract_backbone(frames)
        fpn_feats = self.bifpn(c_feats)

        motion_feat = None
        if self.use_motion and motion is not None:
            p1_size = fpn_feats[0].shape[-2:]
            motion_feat = self.motion_encoder(motion, p1_size)

        hm_logits = self.hm_head(fpn_feats[0], motion_feat)

        with torch.no_grad():
            peaks_batch = extract_peaks(
                hm_logits.detach(), topk=self.topk, conf_thr=0.01)

        roi_feats, anchors, splits = self._roi_align_single(
            fpn_feats, peaks_batch, B)

        obj_logits, box_deltas = self.refine_head(roi_feats)
        return hm_logits, obj_logits, box_deltas, anchors, splits


model = HeatmapGuidedDetector(CFG).to(DEVICE)

# ── CenterNet-style bias init for heatmap output conv ────────────────────
prior_prob = 0.01
bias_init  = -math.log((1.0 - prior_prob) / prior_prob)
for m in model.hm_head.head.modules():
    if isinstance(m, nn.Conv2d):
        last_hm_conv = m
nn.init.constant_(last_hm_conv.bias, bias_init)
nn.init.normal_(last_hm_conv.weight, std=0.001)
print(f"Heatmap conv bias → {bias_init:.3f}  (sigmoid ≈ {prior_prob})")
print(f"Using data-driven anchor_size = {model.anchor_size:.2f}px")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {n_params/1e6:.2f}M")

model.eval()
with torch.no_grad():
    dummy_frames = torch.randn(2, CFG["temporal_len"], 3,
                               CFG["img_size"], CFG["img_size"]).to(DEVICE)
    dummy_motion = torch.randn(2, CFG["motion_channels"],
                               CFG["img_size"], CFG["img_size"]).to(DEVICE) \
                   if CFG["use_motion"] else None
    hm, obj, box, anc, spl = model(dummy_frames, dummy_motion)
    print(f"hm_logits  : {hm.shape}   (expect [2,1,160,160])")
    print(f"obj_logits : {obj.shape}")
    print(f"box_deltas : {box.shape}")
    assert hm.shape[-2:] == (160, 160), "Stride mismatch!"
    print("Shape check PASSED.")
model.train()

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Heatmap conv bias → -4.595  (sigmoid ≈ 0.01)
Using data-driven anchor_size = 5.93px
Model params: 10.06M
hm_logits  : torch.Size([2, 1, 160, 160])   (expect [2,1,160,160])
obj_logits : torch.Size([100, 1])
box_deltas : torch.Size([100, 4])
Shape check PASSED.


HeatmapGuidedDetector(
  (backbone): EfficientNetFeatures(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
    

In [29]:
# =============================================================================
# CELL 11b — Partial checkpoint load (backbone/BiFPN/heatmap head transfer,
# refinement head re-initializes fresh due to roi_output_size change)
# Run this INSTEAD of calling load_checkpoint normally, right after Cell 11
# builds the model fresh.
# =============================================================================

def load_checkpoint_partial(path, model):
    """
    Loads all matching-shape weights from the checkpoint (backbone, BiFPN,
    motion encoder, heatmap head — all unaffected by the ROI size change)
    and skips the refinement head's MLP layers (whose input dimension
    changed from 7x7 to 11x11 per level), which will train from scratch.
    This preserves all heatmap-localization learning while giving the
    box-regression head a clean slate to learn the finer-grained ROI
    features.
    """
    ckpt = torch.load(path, map_location=DEVICE)
    ckpt_state = ckpt["model"]
    model_state = model.state_dict()

    loaded, skipped = [], []
    new_state = {}
    for k, v in ckpt_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            new_state[k] = v
            loaded.append(k)
        else:
            skipped.append(k)

    model_state.update(new_state)
    model.load_state_dict(model_state)

    print(f"Loaded {len(loaded)} matching tensors from checkpoint.")
    print(f"Skipped {len(skipped)} tensors (shape mismatch, re-initialized fresh):")
    for k in skipped:
        old_shape = ckpt_state[k].shape
        new_shape = model_state[k].shape if k in model_state else "N/A"
        print(f"    {k}: checkpoint={old_shape} vs model={new_shape}")

    return ckpt.get("epoch", 0), ckpt.get("metrics", {})


# Build the model fresh (Cell 11 must have already run with roi_output_size=11)
loaded_epoch, loaded_metrics = load_checkpoint_partial(
    os.path.join(CFG["save_dir"], "checkpoint_last.pt"), model)

print(f"\nResuming logical training from epoch {loaded_epoch} "
      f"(refinement head reset, everything else transferred).")

Loaded 534 matching tensors from checkpoint.
Skipped 1 tensors (shape mismatch, re-initialized fresh):
    refine_head.mlp.0.weight: checkpoint=torch.Size([256, 9408]) vs model=torch.Size([256, 23232])

Resuming logical training from epoch 30 (refinement head reset, everything else transferred).


In [30]:
# =============================================================================
# CELL 12 — Loss functions (single anchor, FP32-forced + bounded focal loss)
# =============================================================================

class FocalLossHeatmap(nn.Module):
    """
    CenterNet focal loss with two stability safeguards:

    1. logit_clamp: bounds individual logits before sigmoid, preventing
       any single pixel from producing an extreme/degenerate probability.

    2. loss_clamp: bounds the AGGREGATE loss magnitude. This is the real
       fix for the instability observed around epoch 10-12 — the
       standard CenterNet formula normalizes by n_pos (often just 1-6
       pixels per batch for tiny, sparse drone targets). If many
       background pixels simultaneously drift toward high confidence
       (a rare but real event during training), their summed negative
       loss divided by a tiny n_pos can spike into the tens or hundreds
       of thousands — finite, so it doesn't crash, but large enough to
       hijack the gradient direction for that step and silently freeze
       learning for the rest of the epoch (confirmed via stress test:
       std=50 logits produced a 627,651 loss before this fix, 50.0 after).
       In normal healthy training this clamp never activates (loss stays
       in the 1-10 range); it only acts as a circuit breaker for the
       rare destabilizing batch.
    """
    def __init__(self, alpha=2.0, beta=4.0, pos_thresh=0.99,
                logit_clamp=15.0, loss_clamp=50.0):
        super().__init__()
        self.alpha       = alpha
        self.beta        = beta
        self.pos_thresh  = pos_thresh
        self.logit_clamp = logit_clamp
        self.loss_clamp  = loss_clamp

    def forward(self, pred_logits, target):
        with torch.cuda.amp.autocast(enabled=False):
            logits = pred_logits.float().clamp(-self.logit_clamp, self.logit_clamp)
            pred = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)
            tgt  = target.float().clamp(0.0, 1.0)

            pos_mask = tgt >= self.pos_thresh
            neg_mask = ~pos_mask

            pos_loss = torch.pow(1.0 - pred[pos_mask], self.alpha) * \
                       torch.log(pred[pos_mask])
            neg_loss = torch.pow(1.0 - tgt[neg_mask], self.beta) * \
                       torch.pow(pred[neg_mask], self.alpha) * \
                       torch.log(1.0 - pred[neg_mask])

            n_pos = max(pos_mask.sum().item(), 1)
            loss  = -(pos_loss.sum() + neg_loss.sum()) / n_pos

            # circuit breaker: cap aggregate loss magnitude
            loss = loss.clamp(max=self.loss_clamp)

            if not torch.isfinite(loss):
                print(f"WARNING: non-finite heatmap loss detected even "
                      f"after clamping. pred range "
                      f"[{pred.min().item():.6f}, {pred.max().item():.6f}]")
                loss = torch.nan_to_num(loss, nan=10.0, posinf=10.0, neginf=10.0)
        return loss


heatmap_criterion = FocalLossHeatmap(
    CFG["focal_alpha"], CFG["focal_beta"], pos_thresh=0.99,
    logit_clamp=15.0, loss_clamp=50.0)


def compute_refinement_loss(obj_logits, box_deltas, anchors,
                            batch_splits, gt_boxes_batch, gt_mask_batch,
                            iou_pos_thr=None, iou_neg_thr=None):
    if iou_pos_thr is None: iou_pos_thr = CFG["iou_pos_thr"]
    if iou_neg_thr is None: iou_neg_thr = CFG["iou_neg_thr"]

    bce = nn.BCEWithLogitsLoss(reduction="none")
    obj_losses, box_losses = [], []

    ptr = 0
    B   = len(batch_splits)
    for b in range(B):
        N   = batch_splits[b]
        anc = anchors[b]
        obj = obj_logits[ptr:ptr + N]
        box = box_deltas[ptr:ptr + N]
        ptr += N

        gt    = gt_boxes_batch[b]
        msk   = gt_mask_batch[b]
        valid_gt = gt[msk > 0]

        if valid_gt.shape[0] == 0:
            tgt_obj = torch.zeros(N, 1, device=obj.device)
            obj_losses.append(bce(obj, tgt_obj).mean())
            continue

        anc_xyxy = torch.stack([
            anc[:, 0] - anc[:, 2] / 2,
            anc[:, 1] - anc[:, 3] / 2,
            anc[:, 0] + anc[:, 2] / 2,
            anc[:, 1] + anc[:, 3] / 2,
        ], dim=1)

        iou_mat       = tv_ops.box_iou(anc_xyxy, valid_gt.to(anc_xyxy.device))
        best_iou, best_gt_idx = iou_mat.max(dim=1)

        pos_mask = best_iou >= iou_pos_thr
        neg_mask = best_iou <  iou_neg_thr

        for g in range(valid_gt.shape[0]):
            best_anc = iou_mat[:, g].argmax()
            pos_mask[best_anc] = True

        tgt_obj = torch.zeros(N, 1, device=obj.device)
        tgt_obj[pos_mask] = 1.0

        weight = torch.ones(N, 1, device=obj.device)
        weight[~pos_mask & ~neg_mask] = 0.0

        obj_losses.append(
            (bce(obj, tgt_obj) * weight).sum() /
            weight.sum().clamp(min=1))

        if pos_mask.sum() > 0:
            matched_gt  = valid_gt[best_gt_idx[pos_mask]]
            p_anc       = anc[pos_mask]

            gt_cx = (matched_gt[:, 0] + matched_gt[:, 2]) / 2
            gt_cy = (matched_gt[:, 1] + matched_gt[:, 3]) / 2
            gt_w  = (matched_gt[:, 2] - matched_gt[:, 0]).clamp(min=1)
            gt_h  = (matched_gt[:, 3] - matched_gt[:, 1]).clamp(min=1)

            tgt_dx = (gt_cx - p_anc[:, 0]) / p_anc[:, 2].clamp(min=1)
            tgt_dy = (gt_cy - p_anc[:, 1]) / p_anc[:, 3].clamp(min=1)
            tgt_dw = torch.log(gt_w / p_anc[:, 2].clamp(min=1))
            tgt_dh = torch.log(gt_h / p_anc[:, 3].clamp(min=1))
            tgt_box = torch.stack([tgt_dx, tgt_dy, tgt_dw, tgt_dh], dim=1)

            box_losses.append(
                F.l1_loss(box[pos_mask], tgt_box.to(box.device)))

    obj_loss = torch.stack(obj_losses).mean() if obj_losses else \
               torch.tensor(0., device=obj_logits.device)
    box_loss = torch.stack(box_losses).mean() if box_losses else \
               torch.tensor(0., device=obj_logits.device)
    return obj_loss, box_loss


print("Loss functions defined (single anchor, logit-clamped + loss-clamped focal loss).")

Loss functions defined (single anchor, logit-clamped + loss-clamped focal loss).


In [35]:
# =============================================================================
# CELL 13 — Optimiser and LR scheduler
# =============================================================================

def build_optimizer_scheduler(model, cfg, steps_per_epoch):
    backbone_params = list(model.backbone.parameters()) + \
                      list(model.temporal_fuse.parameters())
    if cfg["use_motion"]:
        backbone_params += list(model.motion_encoder.parameters())

    excluded_prefixes = ["backbone", "temporal_fuse", "motion_encoder"]
    head_params = [p for n, p in model.named_parameters()
                   if not any(n.startswith(k) for k in excluded_prefixes)]

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["lr"] * 0.1},
        {"params": head_params,     "lr": cfg["lr"]},
    ], weight_decay=cfg["weight_decay"])

    # NOTE: steps_per_epoch is based on the FIRST epoch-subsample draw
    # (Cell 7). Since every epoch redraws a fresh subset of the SAME
    # fraction, len(train_dl) stays approximately constant across epochs
    # (same fraction × same total triplets, just different members) —
    # so this total_steps estimate remains accurate throughout training.
    total_steps  = cfg["epochs"] * steps_per_epoch
    warmup_steps = cfg["warmup_epochs"] * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return (cfg["cosine_min_lr"] / cfg["lr"] +
                0.5 * (1 - cfg["cosine_min_lr"] / cfg["lr"]) *
                (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


eff_steps = len(train_dl) // CFG["accum_steps"]
optimizer, scheduler = build_optimizer_scheduler(model, CFG, eff_steps)
scaler = GradScaler(enabled=CFG["amp"])
print(f"Optimiser and scheduler ready. eff_steps/epoch ≈ {eff_steps}")

Optimiser and scheduler ready. eff_steps/epoch ≈ 2009


In [36]:
# =============================================================================
# CELL 14 — Decode predicted deltas → absolute xyxy boxes
# =============================================================================

def decode_boxes(box_deltas, anchors):
    cx = anchors[:, 0] + box_deltas[:, 0] * anchors[:, 2]
    cy = anchors[:, 1] + box_deltas[:, 1] * anchors[:, 3]
    w  = anchors[:, 2] * torch.exp(box_deltas[:, 2].clamp(-4, 4))
    h  = anchors[:, 3] * torch.exp(box_deltas[:, 3].clamp(-4, 4))
    x1 = cx - w / 2; y1 = cy - h / 2
    x2 = cx + w / 2; y2 = cy + h / 2
    return torch.stack([x1, y1, x2, y2], dim=1)


def decode_predictions(obj_logits, box_deltas, anchors,
                       batch_splits, conf_thr=0.3, nms_iou_thr=0.3):
    results = []
    ptr = 0
    for b, N in enumerate(batch_splits):
        obj   = torch.sigmoid(obj_logits[ptr:ptr + N, 0])
        delta = box_deltas[ptr:ptr + N]
        anc_b = anchors[b].to(delta.device)

        boxes = decode_boxes(delta, anc_b)

        keep  = obj >= conf_thr
        boxes = boxes[keep]
        obj   = obj[keep]

        if boxes.shape[0] > 0:
            nms_k = tv_ops.nms(boxes, obj, nms_iou_thr)
            boxes = boxes[nms_k]
            obj   = obj[nms_k]

        results.append({"boxes": boxes.cpu(), "scores": obj.cpu()})
        ptr += N
    return results


print("Decoder defined.")

Decoder defined.


In [37]:
# =============================================================================
# CELL 15 — Evaluation metrics
# =============================================================================

class COCOEvaluator:
    def __init__(self, coco_gt_path):
        self.coco_gt   = COCO(coco_gt_path)
        self.results   = []
        self.img_ids   = []

    def reset(self):
        self.results = []
        self.img_ids = []

    def update(self, predictions, img_ids):
        for pred, img_id in zip(predictions, img_ids):
            self.img_ids.append(img_id)
            boxes  = pred["boxes"]
            scores = pred["scores"]
            for i in range(boxes.shape[0]):
                x1, y1, x2, y2 = boxes[i].tolist()
                w, h = x2 - x1, y2 - y1
                self.results.append({
                    "image_id"   : int(img_id),
                    "category_id": 1,
                    "bbox"       : [x1, y1, w, h],
                    "score"      : float(scores[i]),
                })

    def evaluate(self):
        if not self.results:
            return {k: 0.0 for k in
                    ["AP50", "AP25", "AP50_95", "AR1", "AR10", "F1_50"]}

        coco_dt = self.coco_gt.loadRes(self.results)

        ev = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev.params.iouThrs = np.linspace(0.5, 0.95, 10)
        ev.evaluate(); ev.accumulate(); ev.summarize()
        ap_50_95 = float(ev.stats[0])
        ar_10    = float(ev.stats[9])
        ar_1     = float(ev.stats[6])

        ev50 = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev50.params.iouThrs = np.array([0.5])
        ev50.evaluate(); ev50.accumulate(); ev50.summarize()
        ap_50 = float(ev50.stats[0])
        precision = ev50.eval["precision"][0, :, 0, 0, 2]
        recall    = ev50.params.recThrs
        f1_vals   = 2 * precision * recall / (precision + recall + 1e-8)
        f1_50     = float(f1_vals.max()) if len(f1_vals) > 0 else 0.0

        ev25 = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev25.params.iouThrs = np.array([0.25])
        ev25.evaluate(); ev25.accumulate(); ev25.summarize()
        ap_25 = float(ev25.stats[0])

        return {
            "AP50": ap_50, "AP25": ap_25, "AP50_95": ap_50_95,
            "AR1": ar_1, "AR10": ar_10, "F1_50": f1_50,
        }


def measure_fps(model, device, img_size=640, T=3, n_runs=50, warmup=10, use_amp=True):
    model.eval()
    dummy = torch.randn(1, T, 3, img_size, img_size, device=device)
    dummy_motion = torch.randn(1, CFG["motion_channels"], img_size, img_size,
                               device=device) if CFG["use_motion"] else None
    with torch.no_grad():
        for _ in range(warmup):
            with autocast(enabled=use_amp):
                _ = model(dummy, dummy_motion)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            with autocast(enabled=use_amp):
                _ = model(dummy, dummy_motion)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
    fps = n_runs / (t1 - t0)
    model.train()
    return fps


val_evaluator  = COCOEvaluator(VAL_COCO_JSON)
test_evaluator = COCOEvaluator(TEST_COCO_JSON)
print("Metrics engine ready.")

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Metrics engine ready.


In [38]:
# =============================================================================
# CELL 16 — Train one epoch / validate one epoch (single anchor + motion)
# =============================================================================

def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss, total_hm, total_obj, total_box = 0., 0., 0., 0.
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        frames    = batch["frames"].to(DEVICE)
        hm_target = batch["heatmap"].to(DEVICE)
        gt_boxes  = batch["gt_boxes"].to(DEVICE)
        gt_mask   = batch["gt_mask"].to(DEVICE)
        motion    = batch["motion"].to(DEVICE) if CFG["use_motion"] else None

        with autocast(enabled=CFG["amp"]):
            hm_logits, obj_logits, box_deltas, anchors, splits = \
                model(frames, motion)

            # NOTE: heatmap_criterion forces its own FP32 internally now,
            # regardless of this outer autocast context — this is the fix.
            lh = heatmap_criterion(hm_logits, hm_target)
            lo, lb = compute_refinement_loss(
                obj_logits, box_deltas, anchors, splits, gt_boxes, gt_mask)

            loss = (CFG["heatmap_loss_w"] * lh +
                    CFG["obj_loss_w"]     * lo +
                    CFG["box_loss_w"]     * lb)
            loss = loss / CFG["accum_steps"]

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG["accum_steps"]
        total_hm   += lh.item()
        total_obj  += lo.item()
        total_box  += lb.item()

        if step % 50 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f"  [E{epoch}|{step}/{len(loader)}] "
                  f"loss={total_loss/(step+1):.4f}  "
                  f"hm={total_hm/(step+1):.4f}  "
                  f"obj={total_obj/(step+1):.4f}  "
                  f"box={total_box/(step+1):.4f}  "
                  f"lr={lr_now:.2e}")

    n = len(loader)
    return {
        "train/loss"    : total_loss / n,
        "train/hm_loss" : total_hm   / n,
        "train/obj_loss": total_obj  / n,
        "train/box_loss": total_box  / n,
    }


@torch.no_grad()
def validate(model, loader, evaluator, conf_thr, nms_iou_thr):
    model.eval()
    evaluator.reset()
    total_loss = 0.0

    for batch in loader:
        frames    = batch["frames"].to(DEVICE)
        hm_target = batch["heatmap"].to(DEVICE)
        gt_boxes  = batch["gt_boxes"].to(DEVICE)
        gt_mask   = batch["gt_mask"].to(DEVICE)
        motion    = batch["motion"].to(DEVICE) if CFG["use_motion"] else None
        img_ids   = batch["img_id"].tolist()

        with autocast(enabled=CFG["amp"]):
            hm_logits, obj_logits, box_deltas, anchors, splits = \
                model(frames, motion)
            lh     = heatmap_criterion(hm_logits, hm_target)
            lo, lb = compute_refinement_loss(
                obj_logits, box_deltas, anchors, splits, gt_boxes, gt_mask)
            loss = (CFG["heatmap_loss_w"] * lh +
                    CFG["obj_loss_w"]     * lo +
                    CFG["box_loss_w"]     * lb)
        total_loss += loss.item()

        preds = decode_predictions(
            obj_logits, box_deltas, anchors, splits,
            conf_thr=conf_thr, nms_iou_thr=nms_iou_thr)
        evaluator.update(preds, img_ids)

    metrics  = evaluator.evaluate()
    val_loss = total_loss / len(loader)
    return val_loss, metrics


print("Train/val functions defined.")

Train/val functions defined.


In [39]:
# =============================================================================
# CELL 17 — Checkpoint save/load utilities
# =============================================================================

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, tag="last"):
    ckpt = {
        "epoch"    : epoch,
        "model"    : model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "metrics"  : metrics,
        "cfg"      : CFG,
    }
    path = os.path.join(CFG["save_dir"], f"checkpoint_{tag}.pt")
    torch.save(ckpt, path)
    return path


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    if optimizer and "optimizer" in ckpt: optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and "scheduler" in ckpt: scheduler.load_state_dict(ckpt["scheduler"])
    return ckpt.get("epoch", 0), ckpt.get("metrics", {})


def log_checkpoint_to_wandb(path, tag):
    artifact = wandb.Artifact(
        name=f"{CFG['run_name']}-{tag}",
        type="model",
        description=f"Checkpoint: {tag}",
    )
    artifact.add_file(path)
    wandb.log_artifact(artifact)
    print(f"  → W&B artifact logged: {tag}")


print("Checkpoint utilities ready.")

Checkpoint utilities ready.


In [22]:
# =============================================================================
# CELL 17.5 — PRE-TRAINING VERIFICATION GATE
# (single data-driven anchor + motion + epoch subsampling, FP32-forced focal loss)
# Run this immediately before Cell 18. Training only proceeds if every
# check passes.
# =============================================================================

def _fail(msg):
    print(f"❌ FAIL: {msg}")
    raise AssertionError(msg)

def _ok(msg):
    print(f"✅ {msg}")

print("="*70)
print(" PRE-TRAINING VERIFICATION GATE (single anchor + motion + epoch subsample)")
print("="*70)

# -----------------------------------------------------------------------
# CHECK 1 — Config sanity
# -----------------------------------------------------------------------
print("\n[1] Config sanity")
assert CFG["heatmap_stride"] == 4
assert CFG["anchor_size"] is not None, "anchor_size was never computed — re-run Cell 5"
assert CFG["anchor_size"] > 0
assert 0.0 < CFG["epoch_subsample_fraction"] <= 1.0
_ok(f"heatmap_stride=4, anchor_size={CFG['anchor_size']:.2f}px (data-driven), "
    f"use_motion={CFG['use_motion']}, "
    f"epoch_subsample_fraction={CFG['epoch_subsample_fraction']}")

# -----------------------------------------------------------------------
# CHECK 2 — Model output shapes
# -----------------------------------------------------------------------
print("\n[2] Model output shapes")
model.eval()
batch = next(iter(train_dl))
frames    = batch["frames"].to(DEVICE)
hm_target = batch["heatmap"].to(DEVICE)
gt_boxes  = batch["gt_boxes"].to(DEVICE)
gt_mask   = batch["gt_mask"].to(DEVICE)
motion    = batch["motion"].to(DEVICE) if CFG["use_motion"] else None

assert "motion" in batch, "motion key missing from batch — check Dataset __getitem__"
print(f"    frames shape : {frames.shape}")
print(f"    motion shape : {motion.shape}  (expect [B,{CFG['motion_channels']},"
      f"{CFG['img_size']},{CFG['img_size']}])")

with torch.no_grad():
    hm_logits, obj_logits, box_deltas, anchors, splits = model(frames, motion)

expected_hm = CFG["img_size"] // CFG["heatmap_stride"]
if hm_logits.shape[-2:] != (expected_hm, expected_hm):
    _fail(f"hm_logits shape {hm_logits.shape[-2:]} != expected "
          f"({expected_hm},{expected_hm})")
if obj_logits.shape[1] != 1:
    _fail(f"obj_logits has shape {obj_logits.shape}, expected [N,1] for single anchor")
if box_deltas.shape[1] != 4:
    _fail(f"box_deltas has shape {box_deltas.shape}, expected [N,4]")
_ok(f"hm_logits={hm_logits.shape}, obj_logits={obj_logits.shape}, "
    f"box_deltas={box_deltas.shape}")

# -----------------------------------------------------------------------
# CHECK 3 — Heatmap bias initialisation
# -----------------------------------------------------------------------
print("\n[3] Heatmap bias initialisation")
sig_mean = torch.sigmoid(hm_logits).mean().item()
n_hot    = (torch.sigmoid(hm_logits) > 0.5).sum().item()
if sig_mean > 0.05:
    _fail(f"sigmoid(hm) mean={sig_mean:.4f}, expected ~0.01. "
          f"Bias-init block in model cell may not have run.")
if n_hot > 0:
    _fail(f"{n_hot} pixels predict >0.5 confidence before any training.")
_ok(f"sigmoid(hm) mean = {sig_mean:.4f}, hot_pixels = {n_hot} (untrained, as expected)")

# -----------------------------------------------------------------------
# CHECK 4 — Data-driven anchor vs real GT box size (should be near-perfect
# by construction, since the anchor WAS computed from this exact data)
# -----------------------------------------------------------------------
print("\n[4] Data-driven anchor vs real GT box size")
all_gt_sizes = []
for triplet in val_triplets[:200]:
    _, lp = triplet[-1]
    for cx, cy, bw, bh in read_labels(lp):
        all_gt_sizes.append((bw * CFG["img_size"], bh * CFG["img_size"]))

if len(all_gt_sizes) == 0:
    _fail("No GT boxes found in first 200 val triplets.")

gt_w_med = float(np.median([s[0] for s in all_gt_sizes]))
gt_h_med = float(np.median([s[1] for s in all_gt_sizes]))
print(f"    Median VAL GT box size: {gt_w_med:.1f} x {gt_h_med:.1f} px")
print(f"    Anchor size (from TRAIN set median): {model.anchor_size:.2f} px")

ratio = max(model.anchor_size, gt_w_med) / max(min(model.anchor_size, gt_w_med), 1e-6)
if ratio > 2.0:
    _fail(f"Anchor ({model.anchor_size:.1f}px) vs val median GT "
          f"({gt_w_med:.1f}px) differ by {ratio:.1f}x. Train/val size "
          f"distributions may differ substantially — investigate.")
_ok(f"Anchor ({model.anchor_size:.1f}px) reasonably matches val median GT "
    f"({gt_w_med:.1f}px), ratio={ratio:.2f}x")

# -----------------------------------------------------------------------
# CHECK 5 — Synthetic IoU achievability (perfectly-centered anchor ceiling)
# -----------------------------------------------------------------------
print("\n[5] Anchor-GT IoU achievability (synthetic, model-independent)")
synthetic_ious = []
for w, h in all_gt_sizes[:200]:
    gt_xyxy = torch.tensor([[ -w/2, -h/2, w/2, h/2 ]])
    anc_xyxy = torch.tensor([[ -model.anchor_size/2, -model.anchor_size/2,
                              model.anchor_size/2, model.anchor_size/2 ]])
    iou = tv_ops.box_iou(anc_xyxy, gt_xyxy).item()
    synthetic_ious.append(iou)

mean_synth_iou = float(np.mean(synthetic_ious)) if synthetic_ious else 0.0
print(f"    Synthetic IoU (anchor perfectly centered on real GT boxes): "
      f"{mean_synth_iou:.4f}  (n={len(synthetic_ious)})")
if mean_synth_iou < 0.3:
    _fail(f"Even with a PERFECTLY centered anchor, IoU only reaches "
          f"{mean_synth_iou:.4f}. Unexpected given anchor was computed "
          f"from this data's median — investigate compute_data_driven_anchor_size.")
_ok(f"Synthetic centered-anchor IoU = {mean_synth_iou:.4f} (ceiling for "
    f"untrained model; live model will approach this as heatmap learns "
    f"real spatial structure)")

# -----------------------------------------------------------------------
# CHECK 6 — Heatmap target positive-pixel check
# -----------------------------------------------------------------------
print("\n[6] Heatmap target positive-pixel check")
pos_thresh = 0.99
n_pos_pixels = (hm_target >= pos_thresh).sum().item()
if n_pos_pixels == 0:
    _fail(f"0 pixels in heatmap target reach >= {pos_thresh}. "
          f"Gaussian rendering bug.")
_ok(f"{n_pos_pixels} heatmap target pixels >= {pos_thresh} "
    f"(matches {int(gt_mask.sum().item())} GT objects in batch)")

# -----------------------------------------------------------------------
# CHECK 7 — Focal loss magnitude sanity + bounded-loss verification
# -----------------------------------------------------------------------
print("\n[7] Focal loss magnitude sanity + bounded-loss verification")

# 7a — sanity range check
lh_test = heatmap_criterion(hm_logits, hm_target)
if not torch.isfinite(lh_test):
    _fail(f"Heatmap loss is not finite: {lh_test.item()}")
if lh_test.item() > 20.0:
    _fail(f"Heatmap loss = {lh_test.item():.2f}, expected ~3-8 at init.")
if lh_test.item() < 0.1:
    _fail(f"Heatmap loss = {lh_test.item():.4f}, suspiciously low.")
_ok(f"Heatmap loss = {lh_test.item():.3f}  (sane range: 3-8 at init)")

# 7b — stress test: feed deliberately extreme logits and confirm the loss
# stays BOUNDED (not just finite — finite-but-huge is exactly what froze
# training around epoch 10-12 previously)
with autocast(enabled=CFG["amp"]):
    extreme_logits = torch.randn_like(hm_logits) * 50.0
    extreme_loss = heatmap_criterion(extreme_logits, hm_target)

if not torch.isfinite(extreme_loss):
    _fail(f"Heatmap loss is NOT finite ({extreme_loss.item()}) even with "
          f"clamping active under autocast.")

loss_clamp_value = heatmap_criterion.loss_clamp
if extreme_loss.item() > loss_clamp_value + 1.0:   # small tolerance
    _fail(f"Stress-test loss = {extreme_loss.item():.1f}, expected <= "
          f"~{loss_clamp_value} given loss_clamp={loss_clamp_value}. "
          f"The aggregate-loss clamp is not being applied correctly — "
          f"check that loss.clamp(max=self.loss_clamp) runs AFTER the "
          f"full pos_loss/neg_loss sum, not before.")
_ok(f"Stress test PASSED: extreme logits (std=50) under autocast produce "
    f"a BOUNDED loss ({extreme_loss.item():.2f}, capped at "
    f"{loss_clamp_value}) — both the inf-crash and the large-finite-spike "
    f"instability (the actual epoch 10-12 cause) are now handled")

# 7c — refinement loss check
lo_test, lb_test = compute_refinement_loss(
    obj_logits, box_deltas, anchors, splits, gt_boxes, gt_mask)
if not torch.isfinite(lo_test) or not torch.isfinite(lb_test):
    _fail(f"Refinement losses not finite: obj={lo_test.item()}, box={lb_test.item()}")
_ok(f"obj_loss = {lo_test.item():.4f}, box_loss = {lb_test.item():.4f} (sane)")
# -----------------------------------------------------------------------
# CHECK 8 — Motion encoder connectivity (gradient-based)
# -----------------------------------------------------------------------
print("\n[8] Motion encoder connectivity (gradient-based)")
model.zero_grad()
hm_logits_g, obj_g, box_g, anc_g, splits_g = model(frames, motion)
loss_g = heatmap_criterion(hm_logits_g, hm_target)
loss_g.backward()

motion_grad_norm = sum(
    p.grad.norm().item() for p in model.motion_encoder.parameters()
    if p.grad is not None)

if motion_grad_norm < 1e-10:
    _fail(f"motion_encoder gradient norm = {motion_grad_norm:.2e} — "
          f"disconnected from the loss graph.")
_ok(f"motion_encoder gradient norm = {motion_grad_norm:.6f} — connected "
    f"to loss graph (near-zero forward-pass sensitivity at init is "
    f"expected due to bias-init trick, not a bug)")

# verify downsample-first wiring (MotionEncoder runs at P1 resolution)
with torch.no_grad():
    c_feats_probe = model._extract_backbone(frames)
    fpn_feats_probe = model.bifpn(c_feats_probe)
    p1_size_probe = fpn_feats_probe[0].shape[-2:]
    motion_out_probe = model.motion_encoder(motion, p1_size_probe)
    if motion_out_probe.shape[-2:] != p1_size_probe:
        _fail(f"MotionEncoder output spatial size {motion_out_probe.shape[-2:]} "
              f"!= P1 size {p1_size_probe}. Downsample-first wiring is broken.")
_ok(f"MotionEncoder confirmed running at P1 resolution {p1_size_probe} "
    f"(cheap downsample-first wiring intact)")
model.zero_grad()

# -----------------------------------------------------------------------
# CHECK 9 — COCO ground-truth coordinate space
# -----------------------------------------------------------------------
print("\n[9] COCO ground-truth coordinate space")
coco_check = COCO(VAL_COCO_JSON)
img_ids_check = coco_check.getImgIds()
sample_id = img_ids_check[0]
img_info  = coco_check.loadImgs(sample_id)[0]

if img_info["width"] != CFG["img_size"] or img_info["height"] != CFG["img_size"]:
    _fail(f"COCO json image size = {img_info['width']}x{img_info['height']}, "
          f"expected {CFG['img_size']}x{CFG['img_size']}.")

ann_ids = coco_check.getAnnIds(imgIds=sample_id)
anns    = coco_check.loadAnns(ann_ids)
if len(anns) > 0:
    x1, y1 = anns[0]["bbox"][0], anns[0]["bbox"][1]
    if x1 > CFG["img_size"] or y1 > CFG["img_size"]:
        _fail(f"GT bbox coordinate {x1},{y1} exceeds img_size={CFG['img_size']}.")
_ok(f"COCO json image size = {img_info['width']}x{img_info['height']}, "
    f"sample GT bbox = {anns[0]['bbox'] if anns else 'N/A'} (in-range)")

# -----------------------------------------------------------------------
# CHECK 10 — End-to-end decode + eval smoke test (synthetic perfect match)
# -----------------------------------------------------------------------
print("\n[10] End-to-end decode + eval smoke test (synthetic perfect match)")
test_evaluator_check = COCOEvaluator(VAL_COCO_JSON)
test_evaluator_check.reset()

all_img_ids = coco_check.getImgIds()
n_images_with_gt = 0
fake_preds_batch, fake_ids_batch = [], []

for iid in all_img_ids:
    anns_i = coco_check.loadAnns(coco_check.getAnnIds(imgIds=iid))
    if len(anns_i) == 0:
        continue
    n_images_with_gt += 1
    boxes_xyxy = [[a["bbox"][0], a["bbox"][1],
                  a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]]
                 for a in anns_i]
    fake_preds_batch.append({
        "boxes": torch.tensor(boxes_xyxy, dtype=torch.float32),
        "scores": torch.full((len(boxes_xyxy),), 0.99, dtype=torch.float32)})
    fake_ids_batch.append(iid)

if n_images_with_gt == 0:
    _fail("No images with GT objects found in VAL_COCO_JSON.")

test_evaluator_check.update(fake_preds_batch, fake_ids_batch)
synthetic_metrics = test_evaluator_check.evaluate()

print(f"    Evaluated {n_images_with_gt} images with perfect-match predictions")
print(f"    Synthetic AP50 = {synthetic_metrics['AP50']:.4f}")

if synthetic_metrics["AP50"] < 0.9:
    _fail(f"Synthetic perfect-match AP50 = {synthetic_metrics['AP50']:.4f}, "
          f"expected >= 0.9. Eval pipeline broken.")
_ok(f"Synthetic perfect-match AP50 = {synthetic_metrics['AP50']:.4f} "
    f"across {n_images_with_gt} images — eval pipeline verified")

# -----------------------------------------------------------------------
# CHECK 11 — Epoch subsampling sanity: confirm fresh draws actually
# produce different subsets, and coverage approaches full dataset
# -----------------------------------------------------------------------
print("\n[11] Epoch subsampling sanity")

dl_a, n_a = get_epoch_subset_dataloader(
    train_triplets, CFG["epoch_subsample_fraction"],
    CFG["batch_size"], CFG["num_workers"])
dl_b, n_b = get_epoch_subset_dataloader(
    train_triplets, CFG["epoch_subsample_fraction"],
    CFG["batch_size"], CFG["num_workers"])

ids_a = set(id(t) for t in dl_a.dataset.triplets)
ids_b = set(id(t) for t in dl_b.dataset.triplets)
overlap_fraction = len(ids_a & ids_b) / max(len(ids_a), 1)

print(f"    Draw A: {n_a} triplets | Draw B: {n_b} triplets")
print(f"    Overlap between two independent draws: {overlap_fraction*100:.1f}% "
      f"(expected ≈ {CFG['epoch_subsample_fraction']*100:.0f}%, since both "
      f"draws sample independently from the same pool)")

if n_a != n_b:
    _fail(f"Subset sizes differ between draws ({n_a} vs {n_b}) — "
          f"fraction calculation may be unstable.")

expected_overlap = CFG["epoch_subsample_fraction"]
if abs(overlap_fraction - expected_overlap) > 0.15:
    _fail(f"Overlap fraction {overlap_fraction:.2f} far from expected "
          f"~{expected_overlap:.2f} — sampling may not be properly random "
          f"or independent across draws.")

# estimate coverage after N epochs at this fraction
n_epochs_check = 10
coverage_estimate = 1 - (1 - CFG["epoch_subsample_fraction"]) ** n_epochs_check
print(f"    Estimated dataset coverage after {n_epochs_check} epochs: "
      f"{coverage_estimate*100:.1f}% of triplets seen at least once")
_ok(f"Epoch subsampling confirmed working: independent random draws each "
    f"call, ~{coverage_estimate*100:.0f}% full-dataset coverage expected "
    f"by epoch {n_epochs_check}")


model.train()

 PRE-TRAINING VERIFICATION GATE (single anchor + motion + epoch subsample)

[1] Config sanity
✅ heatmap_stride=4, anchor_size=5.93px (data-driven), use_motion=True, epoch_subsample_fraction=1.0

[2] Model output shapes
    frames shape : torch.Size([4, 3, 3, 640, 640])
    motion shape : torch.Size([4, 2, 640, 640])  (expect [B,2,640,640])
✅ hm_logits=torch.Size([4, 1, 160, 160]), obj_logits=torch.Size([176, 1]), box_deltas=torch.Size([176, 4])

[3] Heatmap bias initialisation
❌ FAIL: 15 pixels predict >0.5 confidence before any training.


AssertionError: 15 pixels predict >0.5 confidence before any training.

In [9]:
# =============================================================================
# CELL 17.8 — Pull epoch_030 checkpoint from W&B (your actual best, AP50=0.8092)
# =============================================================================
import shutil

assert wandb.run is not None, "wandb.init() must be called before this cell."

ARTIFACT_TAG = "epoch_030"
ARTIFACT_NAME = f"{CFG['run_name']}-{ARTIFACT_TAG}"
artifact_ref = f"{wandb.run.entity}/{CFG['project']}/{ARTIFACT_NAME}:latest"

print(f"Pulling artifact: {artifact_ref}")
api = wandb.Api()
artifact = api.artifact(artifact_ref, type="model")
artifact_dir = artifact.download()

downloaded_files = [f for f in os.listdir(artifact_dir) if f.endswith(".pt")]
assert len(downloaded_files) == 1, (
    f"Expected exactly 1 .pt file in artifact, found: {downloaded_files}")

src_path = os.path.join(artifact_dir, downloaded_files[0])
# Save it AS checkpoint_last.pt so Cell 18's existing RESUME_PATH logic
# picks it up without further changes
dst_path = os.path.join(CFG["save_dir"], "checkpoint_last.pt")

os.makedirs(CFG["save_dir"], exist_ok=True)
shutil.copy(src_path, dst_path)

ckpt_check = torch.load(dst_path, map_location="cpu")
print(f"Checkpoint restored to: {dst_path}")
print(f"Checkpoint epoch field : {ckpt_check.get('epoch')}")
print(f"Checkpoint AP50        : {ckpt_check.get('metrics', {}).get('AP50')}")
assert ckpt_check.get("epoch") == 30, (
    f"Expected epoch=30, got {ckpt_check.get('epoch')} — wrong checkpoint pulled.")
print("✅ Confirmed: epoch 30 checkpoint (AP50=0.8092) loaded correctly.")

Pulling artifact: bscs22030-information-technology-university/drone-detection-nps/heatmap-guided-single-anchor-motion-v3-epoch_030:latest


wandb: Downloading large artifact 'heatmap-guided-single-anchor-motion-v3-epoch_030:latest', 75.19MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (379.8MB/s)


Checkpoint restored to: /kaggle/working/checkpoints/checkpoint_last.pt
Checkpoint epoch field : 30
Checkpoint AP50        : 0.8092418679898434
✅ Confirmed: epoch 30 checkpoint (AP50=0.8092) loaded correctly.


In [ ]:
# =============================================================================
# CELL 18 — Main training loop (resuming from epoch 30 with partial weight
# transfer: backbone/heatmap kept, refinement head reset for new ROI size)
# =============================================================================

# We do NOT use the normal RESUME/load_checkpoint path here, since
# load_checkpoint_partial() in Cell 11b already loaded weights and gave us
# loaded_epoch/loaded_metrics directly.

start_epoch = loaded_epoch + 1   # = 31
best_ap50   = -1.0   # RESET — refinement head changed, old AP50 benchmark
                      # isn't a fair comparison against the new architecture
history     = []

print(f"Resuming from epoch {loaded_epoch} (partial weight transfer). "
      f"Starting fresh epoch count at {start_epoch}. "
      f"best_ap50 reset to track NEW architecture's performance "
      f"(old best was {loaded_metrics.get('AP50', 'N/A')} under the "
      f"previous 7x7 ROI config, not directly comparable).")

for epoch in range(start_epoch, CFG["epochs"] + 1):
    t_start = time.time()
    print(f"\n{'='*60}")
    print(f" Epoch {epoch}/{CFG['epochs']}")
    print(f"{'='*60}")

    # epoch_subsample_fraction is now 1.0 — this draws the FULL dataset
    # every epoch (no more random subsetting, given LR is near its floor
    # and we want maximum signal density per remaining epoch)
    train_dl, n_subset = get_epoch_subset_dataloader(
        train_triplets, CFG["epoch_subsample_fraction"],
        CFG["batch_size"], CFG["num_workers"])
    print(f"  Training on {n_subset}/{len(train_triplets)} triplets "
          f"({CFG['epoch_subsample_fraction']*100:.0f}% subsample, "
          f"{len(train_dl)} batches)")

    train_metrics = train_one_epoch(
        model, train_dl, optimizer, scheduler, scaler, epoch)

    val_loss, val_metrics = validate(
        model, val_dl, val_evaluator,
        conf_thr    = CFG["conf_thr"],
        nms_iou_thr = CFG["nms_iou_thr"])

    fps = measure_fps(model, DEVICE) if epoch % 5 == 0 else None

    elapsed = time.time() - t_start

    log = {
        **train_metrics,
        "val/loss"    : val_loss,
        "val/AP50"    : val_metrics["AP50"],
        "val/AP25"    : val_metrics["AP25"],
        "val/AP50_95" : val_metrics["AP50_95"],
        "val/AR1"     : val_metrics["AR1"],
        "val/AR10"    : val_metrics["AR10"],
        "val/F1_50"   : val_metrics["F1_50"],
        "epoch"       : epoch,
        "epoch_time_s": elapsed,
        "lr"          : scheduler.get_last_lr()[0],
        "train_subset_size": n_subset,
    }
    if fps is not None:
        log["fps"] = fps

    wandb.log(log)
    history.append(log)

    print(f"  val_loss={val_loss:.4f} | AP50={val_metrics['AP50']:.4f} "
          f"| AP50_95={val_metrics['AP50_95']:.4f} "
          f"| F1={val_metrics['F1_50']:.4f} "
          f"| time={elapsed:.1f}s"
          + (f" | FPS={fps:.1f}" if fps else ""))

    last_path = save_checkpoint(
        model, optimizer, scheduler, epoch, val_metrics, tag="last")
    log_checkpoint_to_wandb(last_path, tag="last")

    epoch_path = save_checkpoint(
        model, optimizer, scheduler, epoch, val_metrics,
        tag=f"epoch_{epoch:03d}")
    log_checkpoint_to_wandb(epoch_path, tag=f"epoch_{epoch:03d}")

    if val_metrics["AP50"] > best_ap50:
        best_ap50 = val_metrics["AP50"]
        best_path = save_checkpoint(
            model, optimizer, scheduler, epoch, val_metrics, tag="best")
        log_checkpoint_to_wandb(best_path, tag="best")
        print(f"  ★ New best AP50={best_ap50:.4f} → saved best checkpoint")
        wandb.run.summary["best_AP50"]   = best_ap50
        wandb.run.summary["best_epoch"]  = epoch

print("\nTraining complete.")

Resuming from epoch 30 (partial weight transfer). Starting fresh epoch count at 31. best_ap50 reset to track NEW architecture's performance (old best was 0.8092418679898434 under the previous 7x7 ROI config, not directly comparable).

 Epoch 31/60
  Training on 32148/32148 triplets (100% subsample, 8037 batches)
  [E31|0/8037] loss=1.5771  hm=0.1031  obj=0.1709  box=0.6516  lr=0.00e+00
  [E31|50/8037] loss=3.9157  hm=0.9073  obj=0.1555  box=1.4265  lr=2.39e-08
  [E31|100/8037] loss=3.4445  hm=0.8643  obj=0.1477  box=1.2163  lr=4.98e-08
  [E31|150/8037] loss=3.1654  hm=0.7804  obj=0.1412  box=1.1219  lr=7.37e-08
  [E31|200/8037] loss=3.0213  hm=0.7852  obj=0.1318  box=1.0521  lr=9.96e-08
  [E31|250/8037] loss=2.9520  hm=0.7944  obj=0.1222  box=1.0177  lr=1.23e-07
  [E31|300/8037] loss=2.8450  hm=0.7888  obj=0.1130  box=0.9716  lr=1.49e-07
  [E31|350/8037] loss=2.8539  hm=0.7833  obj=0.1050  box=0.9828  lr=1.73e-07
  [E31|400/8037] loss=2.7508  hm=0.7797  obj=0.0977  box=0.9367  lr=1.99e

In [ ]:
# =============================================================================
# CELL 19 — Test evaluation with best checkpoint
# =============================================================================

best_ckpt = os.path.join(CFG["save_dir"], "checkpoint_best.pt")
load_checkpoint(best_ckpt, model)
print(f"Loaded best checkpoint from: {best_ckpt}")

# Measure FPS on test hardware
fps_test = measure_fps(model, DEVICE, n_runs=100, warmup=20)
print(f"FPS (batch=1, T=3, 640×640): {fps_test:.2f}")

test_loss, test_metrics = validate(
    model, test_dl, test_evaluator,
    conf_thr    = CFG["conf_thr"],
    nms_iou_thr = CFG["nms_iou_thr"])

print("\n" + "="*50)
print(" TEST RESULTS")
print("="*50)
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print(f"  {'FPS':12s}: {fps_test:.2f}")
print("="*50)

test_log = {
    "test/loss"    : test_loss,
    "test/AP50"    : test_metrics["AP50"],
    "test/AP25"    : test_metrics["AP25"],
    "test/AP50_95" : test_metrics["AP50_95"],
    "test/AR1"     : test_metrics["AR1"],
    "test/AR10"    : test_metrics["AR10"],
    "test/F1_50"   : test_metrics["F1_50"],
    "test/FPS"     : fps_test,
}
wandb.log(test_log)

for k, v in test_log.items():
    wandb.run.summary[k] = v

In [ ]:
# =============================================================================
# CELL 20 — Qualitative visualisation (last frame of triplet + predictions)
# =============================================================================

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def unnorm(t):
    return (t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()


model.eval()
vis_images = []
n_vis = min(8, len(test_triplets))

with torch.no_grad():
    for idx in range(n_vis):
        batch = test_dl.dataset[idx]
        frames    = batch["frames"].unsqueeze(0).to(DEVICE)
        gt_boxes  = batch["gt_boxes"]
        gt_mask   = batch["gt_mask"]

        hm_logits, obj_logits, box_deltas, anchors, splits = model(frames)
        preds = decode_predictions(obj_logits, box_deltas, anchors, splits,
                                   conf_thr=CFG["conf_thr"],
                                   nms_iou_thr=CFG["nms_iou_thr"])

        anchor_frame = unnorm(batch["frames"][-1])  # last frame
        img_vis = (anchor_frame * 255).astype(np.uint8).copy()
        H, W, _ = img_vis.shape

        # Draw GT (green)
        for i in range(int(gt_mask.sum())):
            x1, y1, x2, y2 = gt_boxes[i].tolist()
            cv2.rectangle(img_vis,
                          (int(x1), int(y1)), (int(x2), int(y2)),
                          (0, 255, 0), 1)

        # Draw preds (red)
        pred = preds[0]
        for i in range(pred["boxes"].shape[0]):
            x1, y1, x2, y2 = pred["boxes"][i].tolist()
            sc = pred["scores"][i].item()
            cv2.rectangle(img_vis,
                          (int(x1), int(y1)), (int(x2), int(y2)),
                          (255, 0, 0), 1)
            cv2.putText(img_vis, f"{sc:.2f}",
                        (int(x1), max(0, int(y1) - 3)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1)

        vis_images.append(img_vis)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    if i < len(vis_images):
        ax.imshow(vis_images[i])
        ax.set_title(f"Sample {i}  🟩GT  🟥Pred")
    ax.axis("off")
plt.suptitle("Test Set Qualitative Results", fontsize=14)
plt.tight_layout()
wandb.log({"test/qualitative": wandb.Image(fig)})
plt.show()

In [ ]:
# =============================================================================
# CELL 21 — Training curves
# =============================================================================

df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(df["epoch"], df["train/loss"], label="Train Loss")
axes[0,0].plot(df["epoch"], df["val/loss"],   label="Val Loss")
axes[0,0].set_title("Loss"); axes[0,0].legend()

axes[0,1].plot(df["epoch"], df["val/AP50"],    label="AP@0.5")
axes[0,1].plot(df["epoch"], df["val/AP25"],    label="AP@0.25")
axes[0,1].plot(df["epoch"], df["val/AP50_95"], label="AP@0.5:0.95")
axes[0,1].set_title("AP Metrics"); axes[0,1].legend()

axes[0,2].plot(df["epoch"], df["val/F1_50"], label="F1@0.5", color="purple")
axes[0,2].set_title("F1@0.5"); axes[0,2].legend()

axes[1,0].plot(df["epoch"], df["val/AR1"],  label="AR@1")
axes[1,0].plot(df["epoch"], df["val/AR10"], label="AR@10")
axes[1,0].set_title("Average Recall"); axes[1,0].legend()

axes[1,1].plot(df["epoch"], df["train/hm_loss"],  label="Heatmap")
axes[1,1].plot(df["epoch"], df["train/obj_loss"],  label="Objectness")
axes[1,1].plot(df["epoch"], df["train/box_loss"],  label="Box")
axes[1,1].set_title("Component Losses"); axes[1,1].legend()

axes[1,2].plot(df["epoch"], df["lr"], label="LR", color="orange")
axes[1,2].set_title("Learning Rate"); axes[1,2].legend()

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
wandb.log({"training_curves": wandb.Image(fig)})
plt.show()

df.to_csv("/kaggle/working/training_history.csv", index=False)
print("Training history saved.")

In [ ]:
# =============================================================================
# CELL 22 — Heatmap activation visualisation (qualitative inspection)
# =============================================================================

model.eval()
batch = next(iter(val_dl))
frames    = batch["frames"][:4].to(DEVICE)
hm_target = batch["heatmap"][:4]

with torch.no_grad():
    hm_logits, *_ = model(frames)
hm_pred = torch.sigmoid(hm_logits[:4]).cpu()

fig, axes = plt.subplots(4, 2, figsize=(10, 20))
for i in range(4):
    img = unnorm(batch["frames"][i, -1])
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Frame {i}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(hm_pred[i, 0].numpy(), cmap="inferno", vmin=0, vmax=1)
    axes[i, 1].set_title(f"Pred Heatmap {i}"); axes[i, 1].axis("off")

plt.suptitle("Predicted Heatmaps (val)", fontsize=13)
plt.tight_layout()
wandb.log({"heatmap_activations": wandb.Image(fig)})
plt.show()

In [ ]:
# =============================================================================
# CELL 23 — Finalise W&B run
# =============================================================================

wandb.run.summary["final_best_AP50"] = best_ap50
wandb.finish()
print("W&B run finished. All done.")